# Crop Price Prediction — Chronos (zero-shot) on Colab

This notebook finishes the comparison in the main project (`crop_price_prediction.py`):
Seasonal Naive vs Linear Regression vs XGBoost — **plus Chronos**, Amazon's pre-trained
time-series model. Chronos needs no training: give it past prices, it forecasts the future.

**Same test as the main project, so the numbers are directly comparable:** last 12 months
(Oct 2025 – Sep 2026) are the test set. Verified results from the main script:

| Model | RMSE | MAPE |
|---|---|---|
| Seasonal Naive | 44.57 | 17.66% |
| Linear Regression | 15.11 | 5.65% |
| XGBoost | 14.81 | 5.68% |

Run this cell-by-cell (or Runtime → Run all). First run downloads the small Chronos
checkpoint (~200 MB, `amazon/chronos-t5-small`) — allow 2–3 minutes. It is free: Colab's
free tier is enough, no GPU needed (CPU runtime works, GPU is just faster).

To try another crop, change `CROP` in the data cell to `"maize"` or `"rice"` and re-run
from that cell down. (For maize/rice the comparison numbers above change — re-run the
main script with the same crop to get its baseline table.)

In [ ]:
# Cell 1 — install Chronos (free, open-source, Apache-2.0)
!pip install -q chronos-forecasting

In [ ]:
# Cell 2 — real wheat prices, downloaded live from FRED (same source as the main project)
import pandas as pd

CROP = "wheat"  # change to "maize" or "rice"
FRED = {
    "wheat": ("https://fred.stlouisfed.org/graph/fredgraph.csv?id=PWHEAMTUSDM", "PWHEAMTUSDM"),
    "maize": ("https://fred.stlouisfed.org/graph/fredgraph.csv?id=PMAIZMTUSDM", "PMAIZMTUSDM"),
    "rice":  ("https://fred.stlouisfed.org/graph/fredgraph.csv?id=PRICENPQUSDM", "PRICENPQUSDM"),
}
url, col = FRED[CROP]
df = pd.read_csv(url)
df["date"] = pd.to_datetime(df["observation_date"])
df["price"] = pd.to_numeric(df[col], errors="coerce")
df = df[["date", "price"]].dropna().sort_values("date").reset_index(drop=True)
print(f"{CROP}: {len(df)} months, {df['date'].iloc[0].date()} to {df['date'].iloc[-1].date()}")
print(df.tail(3).to_string(index=False))

In [ ]:
# Cell 3 — Chronos zero-shot test on the same last-12-months window
import numpy as np
import torch
from chronos import ChronosPipeline

torch.manual_seed(42)  # Chronos samples, so fix the seed for reproducible numbers

TEST_MONTHS = 12
prices = df["price"].values.astype(np.float32)
actual_test = prices[-TEST_MONTHS:]
context = torch.tensor(prices[:-TEST_MONTHS])

print("Loading amazon/chronos-t5-small (first run downloads ~200 MB)...")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small", device_map="cpu", torch_dtype=torch.float32
)

forecast = pipeline.predict(context, prediction_length=TEST_MONTHS, num_samples=20)
pred = np.median(forecast.numpy(), axis=1).reshape(-1).astype(np.float64)  # median over the 20 samples

mae = float(np.mean(np.abs(actual_test - pred)))
rmse = float(np.sqrt(np.mean((actual_test - pred) ** 2)))
mape = float(np.mean(np.abs((actual_test - pred) / actual_test)) * 100)
print(f"\nChronos (t5-small, zero-shot) — MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%")
print("Compare (wheat): Seasonal Naive RMSE 44.57 | Linear Regression 15.11 | XGBoost 14.81")
print("Lowest RMSE wins. If Chronos beats 14.81, it is the best model for wheat.")

out = pd.DataFrame({
    "date": df["date"].iloc[-TEST_MONTHS:].values,
    "actual": actual_test,
    "chronos": np.round(pred, 2),
})
out.to_csv(f"predictions_{CROP}_chronos.csv", index=False)
print(f"Saved: predictions_{CROP}_chronos.csv")

In [ ]:
# Cell 4 — 6-month forecast from the full history (same horizon as the main project)
full_context = torch.tensor(prices)
forecast6 = pipeline.predict(full_context, prediction_length=6, num_samples=20)
pred6 = np.median(forecast6.numpy(), axis=1).reshape(-1).astype(np.float64)
dates6 = pd.date_range(df["date"].iloc[-1] + pd.offsets.MonthBegin(1), periods=6, freq="MS")
fc = pd.DataFrame({"date": dates6, "forecast_price": np.round(pred6, 2)})
fc.to_csv(f"forecast_next6_{CROP}_chronos.csv", index=False)
print(f"Next 6 months (Chronos, {CROP}):")
print(fc.to_string(index=False))
print(f"Saved: forecast_next6_{CROP}_chronos.csv")

In [ ]:
# Cell 5 — chart: history + Chronos test predictions + 6-month forecast
import matplotlib.pyplot as plt

test_dates = df["date"].iloc[-TEST_MONTHS:]
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["price"], label="Actual price", color="#1f77b4")
plt.plot(test_dates, pred, "o--", label="Chronos (test)", color="#9467bd")
plt.plot(dates6, pred6, "D-", label="Forecast (Chronos)", color="#d62728")
plt.title(f"{CROP.capitalize()} price — Chronos zero-shot (USD per metric ton)")
plt.xlabel("Date")
plt.ylabel("USD / metric ton")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"chart_{CROP}_chronos.png", dpi=150)
plt.show()
print(f"Saved: chart_{CROP}_chronos.png")

## What to do with the result

1. Copy the Chronos MAE / RMSE / MAPE into the results table in `PROJECT_PLAN.md`, next to
   Seasonal Naive, Linear Regression and XGBoost. Lowest RMSE wins.
2. If Chronos wins, use its 6-month forecast (`forecast_next6_wheat_chronos.csv`) in the
   report's conclusion; if XGBoost still wins, keep XGBoost's forecast and say so honestly
   — reporting that the pre-trained model did *not* beat the trained one is also a good result.
3. Download the two CSVs and the chart from Colab (left sidebar → Files) into the project
   folder, next to the other `predictions_*.csv` / `chart_*.png` files.

Note: this notebook exists because a local run was tried on 2026-11-13 and the model
download was blocked in that background environment (the checkpoint cache was incomplete).
That is an environment limit, not a Chronos problem — Colab downloads the same checkpoint fine.